# Online Softmax — Exercise

Companion to [Attention Part 4 § Online Softmax](https://tanulsingh.github.io/blog/flash-attention#online-softmax-the-trick-that-enables-tiling).

Standard softmax needs the entire row before producing any output (because the denominator sums all values). This is what forces materializing the N×N attention matrix.

**Online softmax processes the row in chunks**, maintaining a running max `m` and running sum `l`. When a new chunk reveals a larger max, all previous values are *retroactively corrected* by multiplying with `exp(m_old - m_new)`. After all chunks, `l` matches what you'd get from batch softmax — you never needed the whole row in memory at once.

## Setup

In [ ]:
# TODO: imports — torch, math

## TODO 1 — `online_softmax`

Given a 1D tensor `x` of arbitrary length, process it in chunks of `chunk_size` and produce the standard softmax output.

For each chunk:
1. Find the chunk's max `m_chunk`.
2. New running max: `m_new = max(m_old, m_chunk)`.
3. Correction factor: `c = exp(m_old - m_new)` (≤ 1).
4. Update running sum: `l_new = l_old * c + sum(exp(x_chunk - m_new))`.

After processing all chunks, the softmax for any element `x_i` is `exp(x_i - m) / l`. To produce the full softmax output, you'll need a second pass with the final `m` and `l` (or store unnormalized exponentials per chunk and normalize at the end).

In [ ]:
def online_softmax(x: 'torch.Tensor', chunk_size: int) -> 'torch.Tensor':
    """
    Args:
        x: 1D tensor of arbitrary length
        chunk_size: how many elements to process at a time
    Returns:
        softmax(x), 1D tensor of same length
    """
    # TODO 1: implement the running-max + correction algorithm
    # Hint:
    #   m = -inf
    #   l = 0
    #   for chunk in chunks:
    #       m_chunk = chunk.max()
    #       m_new = max(m, m_chunk)
    #       l = l * exp(m - m_new) + sum(exp(chunk - m_new))
    #       m = m_new
    #   # second pass: produce normalized output
    #   return exp(x - m) / l
    pass

In [ ]:
# Sanity check:
#   - Reference: torch.softmax(x, dim=-1)
#   - For random x of length 100, online_softmax(x, chunk_size=10) should match the reference
#   - Try chunk_size = 1, 5, 50, 100 — all should give identical results

## TODO 2 — Verify across many random inputs

For 100 random vectors of varying lengths and chunk sizes, assert `torch.allclose(online_softmax(x, c), torch.softmax(x, dim=-1), atol=1e-6)`.

In [ ]:
# TODO 2:
#   - For 100 trials:
#       n = random length, x = random tensor
#       chunk = random divisor of n (or use any chunk and pad)
#       Assert online matches batch
#   - This is the property that makes Flash Attention exact, not approximate

## Run the tests

In [ ]:
from tests import run_online_softmax_tests
# run_online_softmax_tests(online_softmax)